# HDFS Log Clustering Experiments
**Goal:** Evaluate whether HDBSCAN clusters of log embeddings align with true incidents (grouped by `block_id`).

We treat the `block_id` grouping as ground truth — every log belonging to the same block is part of the same incident.
A good clustering should produce clusters where the majority of logs share the same `block_id`.

---
### Experiment plan
1. Load & prepare data (using the same `parsed_logs_1000.json` from exploration)
2. Generate embeddings (TF-IDF as fast baseline, then sentence-transformers)
3. Run HDBSCAN with a sweep of key hyperparameters
4. Evaluate cluster purity against `block_id` ground truth
5. Visualise clusters in 2D (UMAP)
6. Inspect representative logs per cluster
7. Summary & recommendations for the full pipeline

---

## Data Verification

In [5]:
import pandas as pd
import numpy as np

data_path = "../../HDFS_v1/preprocessed/"
# 1. Anomaly labels
labels = pd.read_csv(data_path+"anomaly_label.csv")
print("Labels")
print(labels.shape)
print(labels["Label"].value_counts())
print(labels.head(3))

# 2. Event occurrence matrix
occ = pd.read_csv(data_path+"Event_occurrence_matrix.csv")
print("Event Occurrences")
print(occ.shape)   # expect (~575k rows, 30 cols)
print(occ.columns.tolist())
print(occ.head(3))

# 3. Event traces (sequence representation)
traces = pd.read_csv(data_path+"Event_traces.csv")
print("Traces")
print(traces.shape)
print(traces.head(3))

# 4. NPZ — check what arrays are inside
npz = np.load(data_path+"HDFS.npz", allow_pickle=True)
print("NPZ")
print(npz.files)   # list the array names
for k in npz.files:
    print(k, npz[k].shape, npz[k].dtype)

# 5. Templates
print("Templates")
templates = pd.read_csv(data_path+"HDFS.log_templates.csv")
print(templates)   # should be ~29 rows: EventId, EventTemplate, Occurrences

Labels
(575061, 2)
Label
Normal     558223
Anomaly     16838
Name: count, dtype: int64
                    BlockId    Label
0  blk_-1608999687919862906   Normal
1   blk_7503483334202473044   Normal
2  blk_-3544583377289625738  Anomaly
Event Occurrences
(575061, 32)
['BlockId', 'Label', 'Type', 'E1', 'E2', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19', 'E20', 'E21', 'E22', 'E23', 'E24', 'E25', 'E26', 'E27', 'E28', 'E29']
                    BlockId    Label  Type  E1  E2   E3  E4  E5  E6  E7  ...  \
0  blk_-1608999687919862906  Success   NaN   0   0  203   0  10   7   0  ...   
1   blk_7503483334202473044  Success   NaN   0   2    1   0   3   0   0  ...   
2  blk_-3544583377289625738     Fail  21.0   0   0  203   0   3   0   0  ...   

   E20  E21  E22  E23  E24  E25  E26  E27  E28  E29  
0    0   10    1   10    0    4   10    0    0    0  
1    0    3    1    3    0    0    3    0    0    0  
2    1    3    1    3    0  

In [7]:
occ = pd.read_csv(data_path+"Event_occurrence_matrix.csv")

# 1. What is the Type column?
print("Type value counts:")
print(occ["Type"].value_counts(dropna=False))

# 2. Confirm Label == Type alignment
print("\nLabel vs Type crosstab:")
print(pd.crosstab(occ["Label"], occ["Type"], dropna=False))

# 3. E29 in anomalies vs normal — the timeout event
print("\nE29 > 0:")
print(occ.groupby("Label")["E29"].apply(lambda x: (x > 0).sum()))

# 4. Overall sparsity
feature_cols = [c for c in occ.columns if c.startswith("E")]
print(f"\nFeature sparsity: {(occ[feature_cols] == 0).mean().mean():.1%} of values are zero")

# 5. Event frequency distribution — which events dominate?
print("\nMean event occurrences per block:")
print(occ[feature_cols].mean().sort_values(ascending=False).round(2))

Type value counts:
Type
NaN     558223
5.0       4167
31.0      3225
3.0       2950
0.0       2809
4.0       1240
1.0        953
21.0       724
7.0        476
12.0       130
8.0         45
9.0         34
16.0        22
13.0        10
18.0         9
22.0         9
19.0         8
20.0         7
11.0         3
17.0         3
10.0         3
24.0         3
27.0         3
25.0         2
28.0         1
23.0         1
30.0         1
Name: count, dtype: int64

Label vs Type crosstab:
Type   0.0   1.0   3.0   4.0   5.0   7.0   8.0   9.0   10.0  11.0  ...  20.0  \
Label                                                              ...         
Fail   2809   953  2950  1240  4167   476    45    34     3     3  ...     7   

Type   21.0  22.0  23.0  24.0  25.0  27.0  28.0  30.0  31.0  
Label                                                        
Fail    724     9     1     3     2     3     1     1  3225  

[1 rows x 26 columns]

E29 > 0:
Label
Fail       46
Success     0
Name: E29, dtype: int64

F

In [8]:
traces = pd.read_csv(data_path+"Event_traces.csv")
print(traces.shape)
print(traces.columns.tolist())
print(traces.head(5).to_string())

(575061, 6)
['BlockId', 'Label', 'Type', 'Features', 'TimeInterval', 'Latency']
                    BlockId    Label  Type                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                Features                      

## 0. Installs (run once)

In [ ]:
# Uncomment if packages are missing
# !pip install hdbscan umap-learn sentence-transformers scikit-learn pandas matplotlib seaborn

## 1. Imports & Config

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from collections import Counter
from itertools import product as iterproduct

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    homogeneity_completeness_v_measure,
)
from sklearn.preprocessing import LabelEncoder

import hdbscan
import umap

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
RANDOM_STATE = 42

# ── Path config ──────────────────────────────────────────────────────────────
DATA_DIR = Path("../data")
LOGS_FILE = DATA_DIR / "parsed_logs_1000.json"   # same file used in exploration

# If you have a larger parsed file, swap it in here:
# LOGS_FILE = DATA_DIR / "parsed_logs_full.json"

# Ground-truth anomaly labels (from Loghub HDFS dataset)
# CSV with columns: BlockId, Label  (Label: Normal / Anomaly)
ANOMALY_LABEL_FILE = DATA_DIR / "anomaly_label.csv"  # optional but recommended

print("All imports OK")

## 2. Load & Prepare Data

In [ ]:
with open(LOGS_FILE) as f:
    logs = json.load(f)

df = pd.DataFrame(logs)
df["timestamp"] = pd.to_datetime(df["timestamp"])

# ── Sanity checks ─────────────────────────────────────────────────────────────
assert df["block_id"].notna().all(), "Some logs are missing block_id — filter or impute first."

print(f"Logs loaded  : {len(df):,}")
print(f"Unique blocks: {df['block_id'].nunique():,}  ← these are our 'true incidents'")
print(f"Time window  : {df['timestamp'].min()}  →  {df['timestamp'].max()}")
print()
print("Logs per block (ground-truth incident size):")
print(df.groupby("block_id").size().describe().round(1))

In [ ]:
# ── Attach anomaly labels if available ────────────────────────────────────────
# The HDFS anomaly_label.csv maps each BlockId to Normal/Anomaly.
# Without this file the experiments still run; evaluation is purely structural.

if ANOMALY_LABEL_FILE.exists():
    anomaly_df = pd.read_csv(ANOMALY_LABEL_FILE)
    anomaly_df.columns = ["block_id", "anomaly_label"]
    df = df.merge(anomaly_df, on="block_id", how="left")
    df["anomaly_label"] = df["anomaly_label"].fillna("Unknown")
    print("Anomaly label distribution (log level):")
    print(df["anomaly_label"].value_counts())
else:
    df["anomaly_label"] = "Unknown"
    print("anomaly_label.csv not found — running without ground-truth anomaly labels.")
    print("Download from: https://github.com/logpai/loghub  (HDFS/anomaly_label.csv)")

In [ ]:
# ── Build the text field we will embed ────────────────────────────────────────
# Strategy: strip the block ID from the message (it is an identifier, not semantic content),
# then prepend the component so the embedding captures "where" the event happened.

def preprocess_message(row: pd.Series) -> str:
    """Return a clean, embed-ready string for one log line."""
    msg = row["message"]
    # Remove the block ID so it doesn't dominate embedding similarity
    if pd.notna(row.get("block_id")):
        msg = msg.replace(row["block_id"], "BLK")
    # Strip IP addresses to avoid overfitting to node addresses
    import re
    msg = re.sub(r"\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}(:\d+)?", "IP", msg)
    component = row.get("component", "")
    return f"{component} {msg}".strip()

df["text"] = df.apply(preprocess_message, axis=1)

print("Example preprocessed texts:")
for t in df["text"].head(3):
    print(" •", t[:120])

## 3. Embedding

We test two embedding strategies:
- **TF-IDF** — fast, good baseline for structured logs with repeated templates
- **Sentence-Transformers** (`all-MiniLM-L6-v2`) — semantic, better for varied messages

Switch `EMBEDDING_METHOD` below to compare.

In [ ]:
# ── Choose embedding method ────────────────────────────────────────────────────
EMBEDDING_METHOD = "tfidf"   # "tfidf" | "sbert"

if EMBEDDING_METHOD == "tfidf":
    print("Using TF-IDF embeddings...")
    vectorizer = TfidfVectorizer(
        max_features=512,
        ngram_range=(1, 2),   # unigrams + bigrams
        sublinear_tf=True,    # log-scale TF to dampen high-freq terms
        min_df=2,             # ignore tokens appearing in < 2 logs
    )
    X = vectorizer.fit_transform(df["text"]).toarray().astype(np.float32)
    print(f"TF-IDF matrix: {X.shape}")

elif EMBEDDING_METHOD == "sbert":
    print("Using Sentence-Transformers (all-MiniLM-L6-v2)...")
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    X = model.encode(
        df["text"].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,  # cosine distance becomes Euclidean after normalisation
    ).astype(np.float32)
    print(f"SBERT embedding matrix: {X.shape}")

else:
    raise ValueError(f"Unknown EMBEDDING_METHOD: {EMBEDDING_METHOD}")

print("Embedding complete.")

## 4. UMAP Dimensionality Reduction

HDBSCAN performs best in lower-dimensional spaces. We reduce to 2D for visualisation
and 10D for clustering (a common trade-off — enough to preserve structure, small enough for density estimation).

In [ ]:
# ── 2D UMAP for visualisation ──────────────────────────────────────────────────
print("Fitting UMAP 2D (for visualisation)...")
reducer_2d = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=RANDOM_STATE,
)
X_2d = reducer_2d.fit_transform(X)
print(f"2D UMAP complete: {X_2d.shape}")

# ── 10D UMAP for clustering ────────────────────────────────────────────────────
print("Fitting UMAP 10D (for clustering)...")
reducer_10d = umap.UMAP(
    n_components=10,
    n_neighbors=15,
    min_dist=0.0,   # min_dist=0 packs points tighter → better density estimation
    metric="euclidean",
    random_state=RANDOM_STATE,
)
X_reduced = reducer_10d.fit_transform(X)
print(f"10D UMAP complete: {X_reduced.shape}")

## 5. HDBSCAN Hyperparameter Sweep

Key HDBSCAN parameters for log clustering:
- **`min_cluster_size`**: minimum logs to form a cluster. Too small → noisy micro-clusters. Too large → lumps different incidents together.
- **`min_samples`**: controls how conservative cluster boundary is. Higher → more noise points.
- **`cluster_selection_method`**: `eom` (excess of mass, default) vs `leaf` (finer-grained clusters). `leaf` is often better when true clusters are small.

In [ ]:
def cluster_purity(labels_true: np.ndarray, labels_pred: np.ndarray) -> float:
    """
    Cluster purity: for each predicted cluster, what fraction of its members
    share the majority true label? Averaged across clusters (weighted by size).
    Noise points (label == -1) are excluded from both numerator and denominator.
    """
    mask = labels_pred != -1
    if mask.sum() == 0:
        return 0.0
    lt = labels_true[mask]
    lp = labels_pred[mask]
    total = len(lp)
    correct = 0
    for cluster_id in np.unique(lp):
        cluster_mask = lp == cluster_id
        majority_count = Counter(lt[cluster_mask]).most_common(1)[0][1]
        correct += majority_count
    return correct / total


def evaluate_clustering(df: pd.DataFrame, cluster_labels: np.ndarray) -> dict:
    """Compute a suite of clustering quality metrics."""
    gt = LabelEncoder().fit_transform(df["block_id"])   # integer-encoded block_id
    n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
    noise_frac = (cluster_labels == -1).mean()

    # Exclude noise from ARI / NMI (they can't handle -1 labels)
    mask = cluster_labels != -1
    if mask.sum() < 2 or n_clusters < 2:
        ari = nmi = hom = com = vm = 0.0
    else:
        ari = adjusted_rand_score(gt[mask], cluster_labels[mask])
        nmi = normalized_mutual_info_score(gt[mask], cluster_labels[mask])
        hom, com, vm = homogeneity_completeness_v_measure(gt[mask], cluster_labels[mask])

    purity = cluster_purity(gt, cluster_labels)

    return {
        "n_clusters": n_clusters,
        "noise_frac": round(noise_frac, 3),
        "purity": round(purity, 3),
        "ari": round(ari, 3),
        "nmi": round(nmi, 3),
        "homogeneity": round(hom, 3),
        "completeness": round(com, 3),
        "v_measure": round(vm, 3),
    }

print("Metric functions defined.")

In [ ]:
# ── Grid search ───────────────────────────────────────────────────────────────
# Adjust ranges based on dataset size. For 1000 logs:
#   min_cluster_size: 2–20 is a sensible range (mean incident size ~10)
#   min_samples: 1–5

param_grid = {
    "min_cluster_size": [3, 5, 7, 10, 15],
    "min_samples": [1, 2, 5],
    "cluster_selection_method": ["eom", "leaf"],
}

results = []

keys = list(param_grid.keys())
values = list(param_grid.values())

for combo in iterproduct(*values):
    params = dict(zip(keys, combo))
    clusterer = hdbscan.HDBSCAN(
        **params,
        metric="euclidean",
        core_dist_n_jobs=-1,
    )
    labels = clusterer.fit_predict(X_reduced)
    metrics = evaluate_clustering(df, labels)
    results.append({**params, **metrics})

results_df = pd.DataFrame(results)

print(f"Ran {len(results_df)} experiments.")
print("\nTop 10 configs by purity:")
results_df.sort_values("purity", ascending=False).head(10).to_string(index=False)

In [ ]:
# Full sorted results table
results_df.sort_values("purity", ascending=False).reset_index(drop=True)

### 5.1 Heatmap — Purity vs Hyperparameters

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, method in zip(axes, ["eom", "leaf"]):
    subset = results_df[results_df["cluster_selection_method"] == method]
    pivot = subset.pivot_table(
        index="min_cluster_size",
        columns="min_samples",
        values="purity",
    )
    sns.heatmap(
        pivot,
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap="YlGn",
        vmin=0,
        vmax=1,
        linewidths=0.5,
    )
    ax.set_title(f"Cluster Purity — method={method}", fontweight="bold")
    ax.set_xlabel("min_samples")
    ax.set_ylabel("min_cluster_size")

plt.suptitle("HDBSCAN Hyperparameter Sweep: Cluster Purity", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Noise fraction heatmap — high noise = many logs unclustered
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, method in zip(axes, ["eom", "leaf"]):
    subset = results_df[results_df["cluster_selection_method"] == method]
    pivot = subset.pivot_table(
        index="min_cluster_size",
        columns="min_samples",
        values="noise_frac",
    )
    sns.heatmap(
        pivot,
        ax=ax,
        annot=True,
        fmt=".2f",
        cmap="OrRd",
        vmin=0,
        vmax=1,
        linewidths=0.5,
    )
    ax.set_title(f"Noise Fraction — method={method}", fontweight="bold")
    ax.set_xlabel("min_samples")
    ax.set_ylabel("min_cluster_size")

plt.suptitle("HDBSCAN: Fraction of Logs Labelled as Noise", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 6. Best Config — Deep Dive

In [ ]:
# ── Pick best config by purity ─────────────────────────────────────────────────
best_row = results_df.sort_values("purity", ascending=False).iloc[0]
print("Best configuration:")
print(best_row.to_string())

# Re-fit with best params
best_clusterer = hdbscan.HDBSCAN(
    min_cluster_size=int(best_row["min_cluster_size"]),
    min_samples=int(best_row["min_samples"]),
    cluster_selection_method=best_row["cluster_selection_method"],
    metric="euclidean",
    core_dist_n_jobs=-1,
    prediction_data=True,   # enables soft cluster membership scores
)
best_labels = best_clusterer.fit_predict(X_reduced)
df["cluster"] = best_labels

print(f"\nClusters formed: {(best_labels != -1).sum()} logs in {best_row['n_clusters']:.0f} clusters, "
      f"{(best_labels == -1).sum()} noise points")

### 6.1 UMAP 2D Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Left: colour by HDBSCAN cluster ───────────────────────────────────────────
ax = axes[0]
unique_clusters = sorted(df["cluster"].unique())
n_c = len([c for c in unique_clusters if c != -1])
palette = cm.get_cmap("tab20", max(n_c, 1))

for cid in unique_clusters:
    mask = df["cluster"] == cid
    color = "lightgray" if cid == -1 else palette(cid % 20)
    label = "Noise" if cid == -1 else f"C{cid}"
    ax.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=[color], s=10, alpha=0.6, label=label, linewidths=0,
    )
ax.set_title("HDBSCAN Clusters (best config)", fontweight="bold")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
if n_c <= 20:
    ax.legend(markerscale=2, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7)

# ── Right: colour by anomaly label (if available) ─────────────────────────────
ax = axes[1]
anomaly_colors = {"Normal": "steelblue", "Anomaly": "crimson", "Unknown": "gray"}
for label, color in anomaly_colors.items():
    mask = df["anomaly_label"] == label
    if mask.any():
        ax.scatter(
            X_2d[mask, 0], X_2d[mask, 1],
            c=color, s=10, alpha=0.5, label=label, linewidths=0,
        )
ax.set_title("Ground Truth — Anomaly Label", fontweight="bold")
ax.set_xlabel("UMAP-1")
ax.set_ylabel("UMAP-2")
ax.legend(markerscale=2)

plt.suptitle("UMAP 2D Projection of Log Embeddings", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 6.2 Per-Cluster Purity Analysis

In [ ]:
cluster_stats = []
gt_col = "block_id"

for cid in sorted(df["cluster"].unique()):
    if cid == -1:
        continue
    subset = df[df["cluster"] == cid]
    n = len(subset)
    top_block, top_count = subset[gt_col].value_counts().iloc[[0]].items().__iter__().__next__()
    purity = top_count / n
    n_unique_blocks = subset[gt_col].nunique()
    anomaly_frac = (subset["anomaly_label"] == "Anomaly").mean() if "Anomaly" in df["anomaly_label"].values else None

    cluster_stats.append({
        "cluster": cid,
        "size": n,
        "n_unique_blocks": n_unique_blocks,
        "purity": round(purity, 3),
        "dominant_block": top_block,
        "anomaly_frac": round(anomaly_frac, 3) if anomaly_frac is not None else None,
    })

cluster_stats_df = pd.DataFrame(cluster_stats).sort_values("purity", ascending=False)
print(cluster_stats_df.to_string(index=False))

In [ ]:
# Purity distribution across clusters
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cluster_stats_df["purity"], bins=20, color="steelblue", edgecolor="black")
axes[0].axvline(cluster_stats_df["purity"].mean(), color="red", linestyle="--",
                label=f'Mean: {cluster_stats_df["purity"].mean():.2f}')
axes[0].set_title("Per-Cluster Purity Distribution", fontweight="bold")
axes[0].set_xlabel("Purity (fraction sharing dominant block_id)")
axes[0].set_ylabel("Number of Clusters")
axes[0].legend()

axes[1].scatter(cluster_stats_df["size"], cluster_stats_df["purity"],
                alpha=0.7, color="coral", edgecolors="black", linewidths=0.5)
axes[1].set_title("Cluster Size vs Purity", fontweight="bold")
axes[1].set_xlabel("Cluster Size (# logs)")
axes[1].set_ylabel("Purity")

plt.tight_layout()
plt.show()

### 6.3 HDBSCAN Condensed Tree

In [ ]:
# The condensed tree shows how clusters merge as lambda (1/distance) increases.
# This is the hierarchical structure that HDBSCAN ultimately selects from.

fig, ax = plt.subplots(figsize=(14, 6))
best_clusterer.condensed_tree_.plot(
    select_clusters=True,
    selection_palette=sns.color_palette("tab20"),
    axis=ax,
)
ax.set_title("HDBSCAN Condensed Cluster Tree", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

## 7. Qualitative Inspection — Sample Logs per Cluster

In [ ]:
N_CLUSTERS_TO_SHOW = 5
N_LOGS_PER_CLUSTER = 4

# Show the N largest clusters (excluding noise)
top_clusters = (
    df[df["cluster"] != -1]
    .groupby("cluster")
    .size()
    .nlargest(N_CLUSTERS_TO_SHOW)
    .index.tolist()
)

for cid in top_clusters:
    subset = df[df["cluster"] == cid]
    dominant_block = subset["block_id"].value_counts().index[0]
    purity = cluster_stats_df.loc[cluster_stats_df["cluster"] == cid, "purity"].values[0]

    print(f"{'='*70}")
    print(f"Cluster {cid}  |  size={len(subset)}  |  purity={purity:.2f}  |  dominant_block={dominant_block}")
    print(f"Unique blocks: {subset['block_id'].nunique()}")
    if "Anomaly" in df["anomaly_label"].values:
        print(f"Anomaly logs: {(subset['anomaly_label'] == 'Anomaly').sum()} / {len(subset)}")
    print()
    for _, row in subset.sample(min(N_LOGS_PER_CLUSTER, len(subset)), random_state=RANDOM_STATE).iterrows():
        print(f"  [{row['component']}]")
        print(f"  {row['message'][:100]}")
        print()

print("="*70)

## 8. Metric Summary Across All Experiments

In [ ]:
# Parallel coordinates plot to compare runs across all metrics
from pandas.plotting import parallel_coordinates

plot_df = results_df.copy()
# Create a readable label for each config
plot_df["config"] = (
    "mcs=" + plot_df["min_cluster_size"].astype(str)
    + " ms=" + plot_df["min_samples"].astype(str)
    + " " + plot_df["cluster_selection_method"]
)

metric_cols = ["purity", "ari", "nmi", "homogeneity", "completeness", "noise_frac"]

fig, ax = plt.subplots(figsize=(14, 6))
parallel_coordinates(
    plot_df[["config"] + metric_cols],
    "config",
    ax=ax,
    alpha=0.4,
    color=plt.cm.viridis(np.linspace(0, 1, len(plot_df))),
)
ax.set_title("All Experiment Configs — Metrics Overview", fontweight="bold")
ax.get_legend().remove()
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print("\nMetric correlations:")
results_df[metric_cols].corr().round(2)

## 9. Embedding Comparison (TF-IDF vs SBERT)

Re-run with `EMBEDDING_METHOD = "sbert"` in Section 3, then paste the best-row metrics here for a side-by-side comparison.
This cell keeps a running record so you don't have to re-run everything.

In [ ]:
# Fill in after running both embedding methods
comparison = pd.DataFrame([
    # {"embedding": "tfidf",  "purity": ..., "ari": ..., "nmi": ..., "noise_frac": ..., "n_clusters": ...},
    # {"embedding": "sbert",  "purity": ..., "ari": ..., "nmi": ..., "noise_frac": ..., "n_clusters": ...},
])

if not comparison.empty:
    print(comparison.to_string(index=False))
else:
    print("Fill in comparison results after running both embedding methods.")

## 10. Export — Cluster Labels for LLM Summarisation Pipeline

This produces the JSON format expected by the downstream LLM summarisation step.

In [ ]:
# Export cluster assignments + embeddings in the format the LLM step will consume

OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(exist_ok=True)

output_records = []
for cid in sorted(df["cluster"].unique()):
    cluster_logs = df[df["cluster"] == cid].copy()
    record = {
        "cluster_id": int(cid),
        "is_noise": cid == -1,
        "n_logs": len(cluster_logs),
        "dominant_block_id": cluster_logs["block_id"].value_counts().index[0] if cid != -1 else None,
        "logs": cluster_logs[["line_number", "timestamp", "component", "block_id", "message"]]
                .assign(timestamp=lambda d: d["timestamp"].astype(str))
                .to_dict("records"),
        # embeddings stored as lists for JSON serialisability
        "embeddings": X_reduced[cluster_logs.index].tolist(),
    }
    output_records.append(record)

out_path = OUTPUT_DIR / f"clusters_{EMBEDDING_METHOD}.json"
with open(out_path, "w") as f:
    json.dump(output_records, f, indent=2)

print(f"Exported {len(output_records)} cluster records to {out_path}")
print(f"  — {sum(1 for r in output_records if not r['is_noise'])} real clusters")
print(f"  — 1 noise pseudo-cluster")

---
## 11. Findings & Recommendations

*(Fill in after running experiments)*

### What to look for

| Metric | What it means for your use case |
|---|---|
| **Purity ↑** | Each cluster is dominated by logs from one incident — ideal for the LLM step |
| **Noise fraction ↓** | Fewer logs thrown away as unclustered |
| **Homogeneity ↑** | Clusters don't mix incidents |
| **Completeness ↑** | A whole incident ends up in one cluster (harder to achieve) |
| **ARI / NMI ↑** | Agreement with ground truth beyond chance |

### Known limitations of this 1000-log window
- Only **17 seconds** of logs — you may see very clean clustering because the log types in this window are limited.
- All logs are `INFO` level — anomaly signal won't appear until you load the full dataset with `anomaly_label.csv`.
- HDBSCAN works best when incident sizes are consistent. The high variance (std ~10.8 on mean ~9.8) will cause small incidents to be absorbed into noise.

### Next steps
1. **Increase data volume** — run on the full HDFS dataset (~11M logs) with a sliding window approach.
2. **Load `anomaly_label.csv`** — evaluate whether anomalous blocks cluster separately from normal ones.
3. **Try SBERT embeddings** — semantic similarity should help separate incidents with similar template text but different operational context.
4. **Consider log drain / log parsing templates** — replacing variable tokens (IPs, block IDs, sizes) with placeholders before embedding often significantly improves cluster purity.
5. **Tune `min_cluster_size` to median incident size** — from exploration, the median was 13 logs; `min_cluster_size=13` is a principled starting point.

---